# DiffDockによるBRD4ドッキング（Google Colab）

DiffDock検証レポート（このフォルダの `README.md`）のドッキング計算を行ったノートブック。

- 実行環境: Google Colab（NVIDIA T4 GPU）
- DiffDock: GitHub `gcorso/DiffDock` のmainブランチ（2026年10月時点）、モデル重みはv1.1（DiffDock-L）
- 事前にアップロードするファイル: `brd4_bd1_simple.pdb`（学習ノートの特徴量抽出で作成）、`3mxf.pdb`
- 各実行の結果はzipでダウンロードし、このフォルダの `diffdock_run1_Risomer/`、`diffdock_run2_Sisomer/`、`diffdock_run3_controls/` に保存している

| 実行 | 化合物 | 出力先 |
|---|---|---|
| 1回目 | JQ1（(R)体で入力していた）、top_pred | `results/brd4_run` |
| 2回目 | JQ1（(S)体に修正） | `results/jq1_s_run` |
| 3回目 | neg_true、fp_highpred | `results/controls_run` |

## 1. 環境構築

公式の`requirements.txt`は古いtorchに固定されているため、Colabのtorchに合わせて依存パッケージを入れる。`openfold`はPDBファイルを入力する場合には不要なため入れていない。

In [ ]:
# GPUが割り当たっているか確認（Noneならランタイムのタイプを変更してください）
!nvidia-smi --query-gpu=name,memory.total --format=csv

# DiffDock本体を取得
!git clone https://github.com/gcorso/DiffDock.git
%cd DiffDock

# Colabの現在のtorchバージョンを確認（次のセルで使う）
import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)

name, memory.total [MiB]
Tesla T4, 15360 MiB
Cloning into 'DiffDock'...
remote: Enumerating objects: 520, done.
remote: Counting objects: 100% (293/293), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 520 (delta 209), reused 154 (delta 154), pack-reused 227 (from 3)
Receiving objects: 100% (520/520), 233.08 MiB | 32.84 MiB/s, done.
Resolving deltas: 100% (244/244), done.
/content/DiffDock/DiffDock
torch: 2.11.0+cu130 | CUDA: 13.0


In [ ]:
import torch, re
v = torch.__version__.split('+')[0]          # 例 2.6.0
cu = torch.version.cuda.replace('.', '')     # 例 124 → cu124
tag = f"torch-{v}+cu{cu}"
print("使用するwheel index:", tag)

# コンパイル済みwheelを PyG の配布元から入れる（ソースビルドを避ける）
!pip install -q torch_scatter torch_cluster -f https://data.pyg.org/whl/{tag}.html

# 残りの依存。openfoldは不要なので fair-esm は [esmfold] を付けない
!pip install -q torch_geometric e3nn==0.5.0 fair-esm prody spyrmsd
!pip install -q "rdkit>=2023.3" scipy networkx pandas pyyaml

# 確認
import importlib
for m in ["torch_scatter","torch_cluster","torch_geometric","e3nn","esm","prody","rdkit"]:
    try:
        importlib.import_module(m); print(f"OK   {m}")
    except Exception as ex:
        print(f"NG   {m}: {type(ex).__name__}: {ex}")


使用するwheel index: torch-2.11.0+cu130
OK   torch_scatter
OK   torch_cluster
OK   torch_geometric
OK   e3nn
OK   esm
OK   prody
OK   rdkit


## 2. 1回目の実行：JQ1とtop_pred

モデルの重みとタンパク質構造を配置し、入力CSVを作って実行する。

In [ ]:
import os, glob, shutil

# ===== 1. モデルの重みを取得（GitHub Releasesから約130MB）=====
if not os.path.exists("workdir/v1.1/score_model"):
    !wget -q --show-progress https://github.com/gcorso/DiffDock/releases/download/v1.1/diffdock_models.zip
    !unzip -q -o diffdock_models.zip -d .
print("workdir の中身:", os.listdir("workdir") if os.path.exists("workdir") else "★ workdirが無い")

# ===== 2. アップロードしたPDBを探して配置 =====
cand = glob.glob("/content/**/brd4_bd1_simple.pdb", recursive=True)
if not cand:
    raise FileNotFoundError("brd4_bd1_simple.pdb が見つかりません。左のフォルダ画面にアップロードしてください")
src = cand[0]
if os.path.abspath(src) != os.path.abspath("brd4_bd1_simple.pdb"):
    shutil.copy(src, "brd4_bd1_simple.pdb")
print("PDB:", src, "→ 行数", sum(1 for _ in open("brd4_bd1_simple.pdb")))

# ===== 3. 入力CSVを作る（JQ1=陽性対照、本命化合物の2件）=====
# ※ここで入力したJQ1のSMILESは(R)体だった。後のセルで(S)体に直して再実行している
# complex_name / protein_path / ligand_description / protein_sequence の4列がDiffDockの既定形式
rows = [
    ("JQ1_control", "brd4_bd1_simple.pdb",
     "CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1", ""),
    ("top_pred",    "brd4_bd1_simple.pdb",
     "CCS(=O)(=O)Cc1ccc(Oc2ccc(F)cc2F)c(-c2cn(C)c(=O)c3cc(-c4cc(C)c(OCCO)c(C)c4)oc23)c1", ""),
]
with open("my_input.csv", "w") as f:
    f.write("complex_name,protein_path,ligand_description,protein_sequence\n")
    for r in rows:
        f.write(",".join(r) + "\n")

print("\n--- my_input.csv ---")
print(open("my_input.csv").read())

# ===== 4. SMILESがRDKitで読めるか事前確認（ここで落とす方が早い）=====
from rdkit import Chem
for name, _, smi, _ in rows:
    m = Chem.MolFromSmiles(smi)
    print(f"{name}: {'OK' if m else '★パース失敗'}  原子数={m.GetNumHeavyAtoms() if m else '-'}")

diffdock_models.zip 100%[===================>] 123.81M   278MB/s    in 0.4s    
workdir の中身: ★ workdirが無い
PDB: /content/brd4_bd1_simple.pdb → 行数 1060

--- my_input.csv ---
complex_name,protein_path,ligand_description,protein_sequence
JQ1_control,brd4_bd1_simple.pdb,CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1,
top_pred,brd4_bd1_simple.pdb,CCS(=O)(=O)Cc1ccc(Oc2ccc(F)cc2F)c(-c2cn(C)c(=O)c3cc(-c4cc(C)c(OCCO)c(C)c4)oc23)c1,

JQ1_control: OK  原子数=31
top_pred: OK  原子数=44


In [ ]:
import os, shutil

# zipは score_model/ と confidence_model/ を直下に展開するため、
# default_inference_args.yaml が期待する workdir/v1.1/ 配下へ移動する
os.makedirs("workdir/v1.1", exist_ok=True)
for d in ["score_model", "confidence_model"]:
    if os.path.isdir(d) and not os.path.isdir(f"workdir/v1.1/{d}"):
        shutil.move(d, f"workdir/v1.1/{d}")

# 確認：設定ファイルが指すパスに実体があるか
import yaml
cfg = yaml.safe_load(open("default_inference_args.yaml"))
for key in ["model_dir", "confidence_model_dir"]:
    p = cfg[key]
    print(f"{key}: {p}")
    print("   存在:", os.path.isdir(p), "| 中身:", os.listdir(p) if os.path.isdir(p) else "-")

model_dir: ./workdir/v1.1/score_model
   存在: True | 中身: ['model_parameters.yml', 'best_ema_inference_epoch_model.pt']
confidence_model_dir: ./workdir/v1.1/confidence_model
   存在: True | 中身: ['model_parameters.yml', 'best_model_epoch75.pt']


In [ ]:
!python inference.py \
    --config default_inference_args.yaml \
    --protein_ligand_csv my_input.csv \
    --out_dir results/brd4_run \
    --samples_per_complex 10

/content/DiffDock/DiffDock/utils/so3.py:59: RuntimeWarning: invalid value encountered in sqrt
  _exp_score_norms = np.sqrt(np.sum(_score_norms**2 * _pdf_vals, axis=1) / np.sum(_pdf_vals, axis=1) / np.pi)
100% 201/201 [01:07<00:00,  2.98it/s]
100% 201/201 [01:39<00:00,  2.02it/s]
Generating ESM language model embeddings
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
Processing 1 of 1 batches (2 sequences)
0it [00:00, ?it/s]/content/DiffDock/DiffDock/datasets/parse_chi.py:91: RuntimeWarning: invalid value encountered in cast
  Y = indices.astype(int)
2it [00:58, 29.00s/it]


In [ ]:
import os, glob, re

out = "results/brd4_run"
print("作業ディレクトリ:", os.getcwd())
print("出力先の存在:", os.path.isdir(out))

if os.path.isdir(out):
    for complex_dir in sorted(os.listdir(out)):
        d = os.path.join(out, complex_dir)
        if not os.path.isdir(d): continue
        files = sorted(os.listdir(d))
        print(f"\n=== {complex_dir} ({len(files)}ファイル) ===")
        # rank1_confidence-0.12.sdf のような名前からスコアを抜く
        for f in files:
            m = re.search(r"rank(\d+)_confidence([-\d.]+)\.sdf", f)
            if m:
                print(f"  rank{m.group(1):>2}  confidence={m.group(2)}")
        if not any("confidence" in f for f in files):
            print("  ファイル一覧:", files)
else:
    print("★ 出力先が無い。実行が完了していない可能性あり")
    print("results以下:", glob.glob("results/**", recursive=True)[:20])

作業ディレクトリ: /content/DiffDock/DiffDock
出力先の存在: True

=== JQ1_control (11ファイル) ===
  rank10  confidence=-3.19
  rank 1  confidence=-1.95
  rank 2  confidence=-2.33
  rank 3  confidence=-2.36
  rank 4  confidence=-2.89
  rank 5  confidence=-2.90
  rank 6  confidence=-2.97
  rank 7  confidence=-3.02
  rank 8  confidence=-3.10
  rank 9  confidence=-3.17

=== top_pred (11ファイル) ===
  rank10  confidence=-3.61
  rank 1  confidence=-0.39
  rank 2  confidence=-0.63
  rank 3  confidence=-0.63
  rank 4  confidence=-1.25
  rank 5  confidence=-1.68
  rank 6  confidence=-2.37
  rank 7  confidence=-2.75
  rank 8  confidence=-3.33
  rank 9  confidence=-3.33


## 3. JQ1を(S)体に直して再実行

1回目のJQ1は(R)体（ほぼ不活性な鏡像異性体）になっていた。3MXFに含まれるJQ1は(S)体なので、陽性対照として(S)体で再実行する。

In [ ]:
# ===== JQ1を正しい立体(S体)に修正して再実行 =====
# 1回目に入力したのは [C@@H] = R体(不活性側)。RCSBの化学成分辞書では、3MXF収載のJQ1は (6S) なので [C@H] が正しい。
rows = [
    ("JQ1_control_S", "brd4_bd1_simple.pdb",
     "CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1", ""),
]
with open("jq1_s_input.csv", "w") as f:
    f.write("complex_name,protein_path,ligand_description,protein_sequence\n")
    for r in rows:
        f.write(",".join(r) + "\n")

# 立体が S になっているか、投入直前に確認
from rdkit import Chem
m = Chem.MolFromSmiles(rows[0][2])
print("CIP:", Chem.FindMolChiralCenters(m, useLegacyImplementation=False), "← (13, 'S') であること")
print(open("jq1_s_input.csv").read())


CIP: [(13, 'S')] ← (13, 'S') であること
complex_name,protein_path,ligand_description,protein_sequence
JQ1_control_S,brd4_bd1_simple.pdb,CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1,



In [ ]:
!python inference.py \
    --config default_inference_args.yaml \
    --protein_ligand_csv jq1_s_input.csv \
    --out_dir results/jq1_s_run \
    --samples_per_complex 10

Generating ESM language model embeddings
Processing 1 of 1 batches (1 sequences)
0it [00:00, ?it/s]/content/DiffDock/DiffDock/datasets/parse_chi.py:91: RuntimeWarning: invalid value encountered in cast
  Y = indices.astype(int)
1it [00:26, 26.46s/it]


In [ ]:
import os, re

def scores(out_dir):
    """rank1_confidence-0.39.sdf のような名前からランクとスコアを取り出す"""
    res = {}
    if not os.path.isdir(out_dir): return res
    for cdir in sorted(os.listdir(out_dir)):
        d = os.path.join(out_dir, cdir)
        if not os.path.isdir(d): continue
        pairs = []
        for f in os.listdir(d):
            m = re.search(r"rank(\d+)_confidence([-\d.]+)\.sdf", f)
            if m: pairs.append((int(m.group(1)), float(m.group(2))))
        res[cdir] = sorted(pairs)
    return res

all_res = {}
all_res.update(scores("results/brd4_run"))     # 1回目（JQ1はR体）
all_res.update(scores("results/jq1_s_run"))    # 2回目（JQ1はS体）

print(f"{'complex':<18}{'rank1':>9}{'rank2':>9}{'rank3':>9}   (全10ポーズの最高/最低)")
for name, pairs in all_res.items():
    if not pairs: continue
    top3 = [f"{s:+.2f}" for _, s in pairs[:3]]
    hi, lo = pairs[0][1], max(s for _, s in pairs)
    print(f"{name:<18}" + "".join(f"{t:>9}" for t in top3) +
          f"   max={max(s for _,s in pairs):+.2f} min={min(s for _,s in pairs):+.2f}")

print("\n--- 比較 ---")
r = all_res.get("JQ1_control", [])
s = all_res.get("JQ1_control_S", [])
if r and s:
    print(f"JQ1 R体(陰性対照相当) rank1 = {r[0][1]:+.2f}")
    print(f"JQ1 S体(本来の陽性対照) rank1 = {s[0][1]:+.2f}")
    print(f"差 = {s[0][1]-r[0][1]:+.2f}")

complex               rank1    rank2    rank3   (全10ポーズの最高/最低)
JQ1_control           -1.95    -2.33    -2.36   max=-1.95 min=-3.19
top_pred              -0.39    -0.63    -0.63   max=-0.39 min=-3.61
JQ1_control_S         +0.63    +0.57    +0.52   max=+0.63 min=+0.26

--- 比較 ---
JQ1 R体(陰性対照相当) rank1 = -1.95
JQ1 S体(本来の陽性対照) rank1 = +0.63
差 = +2.58


## 4. 陽性対照のRMSD

結晶構造中のJQ1と、DiffDockの予測ポーズのRMSDを、重ね合わせを行わずに測る。PDBファイルには結合次数の情報がないため、最初のセルでは原子の対応づけに失敗する。次のセルで、SMILESをテンプレートにして結合次数を割り当ててから計算する。

In [ ]:
import os, glob
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign

# ===== 1. 3mxf.pdb から結晶中のJQ1を抜き出す =====
src = glob.glob("/content/**/3mxf.pdb", recursive=True)
if not src:
    raise FileNotFoundError("3mxf.pdb をアップロードしてください")
pdb = src[0]
print("使用PDB:", pdb)

# HETATMのうち残基名がJQ1のものだけ（AチェーンのaltLoc ' ' または 'A'）
lig_lines = []
for l in open(pdb):
    if l.startswith("HETATM") and l[17:20].strip() == "JQ1" and l[21] == "A" and l[16] in (" ", "A"):
        lig_lines.append(l)
print(f"結晶中JQ1の原子数(水素除く): {len(lig_lines)}")
open("jq1_crystal.pdb", "w").write("".join(lig_lines) + "END\n")

ref = Chem.MolFromPDBFile("jq1_crystal.pdb", removeHs=True, sanitize=False)
print("結晶JQ1 読み込み:", "OK" if ref else "★失敗")

# ===== 2. DiffDockのrank1ポーズを読む =====
cand = glob.glob("results/jq1_s_run/JQ1_control_S/rank1_confidence*.sdf")
if not cand:
    cand = glob.glob("results/jq1_s_run/*/rank1*.sdf")
print("予測ポーズ:", cand[0])
pred = Chem.MolFromMolFile(cand[0], removeHs=True, sanitize=False)
print("予測ポーズ 読み込み:", "OK" if pred else "★失敗")
print(f"原子数  結晶={ref.GetNumAtoms()}  予測={pred.GetNumAtoms()}")

# ===== 3. 座標系が同じか（重心の距離）=====
import numpy as np
c_ref = ref.GetConformer().GetPositions().mean(axis=0)
c_prd = pred.GetConformer().GetPositions().mean(axis=0)
print(f"\n重心 結晶={np.round(c_ref,2)}")
print(f"重心 予測={np.round(c_prd,2)}")
print(f"重心間距離 = {np.linalg.norm(c_ref-c_prd):.2f} Å")
print("  (数十Å離れていたら、そもそも別の場所に結合させている)")

# ===== 4. RMSD（重ね合わせなし＝その場での一致度）=====
# 結合ポーズの評価では「重ね合わせずに」測るのが正しい。
# 重ね合わせてしまうと、全く別の場所でも形が同じなら0になってしまう。
try:
    rmsd = rdMolAlign.CalcRMS(pred, ref)     # 対称性を考慮、座標はそのまま
    print(f"\n★ RMSD (in-place, 対称性考慮) = {rmsd:.2f} Å")
    print("   < 2.0 Å: 成功（ドッキング分野の慣例的な基準）")
    print("   2-5 Å  : 部分的に当たっている")
    print("   > 5 Å  : 別の結合様式")
except Exception as e:
    print("CalcRMS 失敗:", e)
    print("→ 原子の対応づけに失敗。SMILESテンプレートを当てる方法に切り替えます")

使用PDB: /content/3mxf.pdb
結晶中JQ1の原子数(水素除く): 31
結晶JQ1 読み込み: OK
予測ポーズ: results/jq1_s_run/JQ1_control_S/rank1_confidence0.63.sdf
予測ポーズ 読み込み: OK
原子数  結晶=31  予測=31

重心 結晶=[28.75 15.83 -2.34]
重心 予測=[28.95 15.71 -2.32]
重心間距離 = 0.23 Å
  (数十Å離れていたら、そもそも別の場所に結合させている)
CalcRMS 失敗: No sub-structure match found between the reference and probe mol
→ 原子の対応づけに失敗。SMILESテンプレートを当てる方法に切り替えます


In [ ]:
import glob, re
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

TEMPLATES = [
    ("中性",    "CC1=C(C)C2=C(S1)N1C(C)=NN=C1[C@H](CC(=O)OC(C)(C)C)N=C2C1=CC=C(Cl)C=C1"),
    ("CCD(+1)", "Cc1sc2c(c1C)C(=N[C@@H](CC(=O)OC(C)(C)C)c3[nH]nc(C)[n+]23)c4ccc(Cl)cc4"),
]

def fix_bonds(raw, quiet=False):
    """結合次数をテンプレートから割り当てる。ref/pred双方に適用して表現を揃える。"""
    for label, smi in TEMPLATES:
        try:
            out = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smi), raw)
            if not quiet: print(f"   テンプレート照合: {label} で成功")
            return out
        except Exception:
            continue
    raise RuntimeError("どのテンプレートでも照合できませんでした")

# ===== 結晶のJQ1 =====
print("結晶JQ1:")
ref = fix_bonds(Chem.MolFromPDBFile("jq1_crystal.pdb", removeHs=True, sanitize=False))

# ===== 予測ポーズ（全10ランク）=====
print("\n結晶構造に対するRMSD（重ね合わせなし・対称性考慮）")
print(f"{'rank':>5}{'confidence':>12}{'RMSD(Å)':>10}   判定")
rows = []
for f in sorted(glob.glob("results/jq1_s_run/JQ1_control_S/rank*_confidence*.sdf")):
    m = re.search(r"rank(\d+)_confidence([-\d.]+)\.sdf", f)
    if not m: continue
    rank, conf = int(m.group(1)), float(m.group(2))
    # predにも同じテンプレートを当てて、ref と表現を揃える
    pred = fix_bonds(Chem.MolFromMolFile(f, removeHs=True, sanitize=False), quiet=True)
    rmsd = rdMolAlign.CalcRMS(pred, ref)
    verdict = "成功(<2Å)" if rmsd < 2 else ("部分的(2-5Å)" if rmsd < 5 else "別の結合様式(>5Å)")
    rows.append((rank, conf, rmsd))
    print(f"{rank:>5}{conf:>12.2f}{rmsd:>10.2f}   {verdict}")

rows.sort()
print(f"\n★ rank1 の RMSD = {rows[0][2]:.2f} Å")
best = min(rows, key=lambda r: r[2])
print(f"   10ポーズ中の最良 = rank{best[0]} の {best[2]:.2f} Å")
print(f"   2Å未満 = {sum(1 for _,_,r in rows if r < 2)} / {len(rows)} ポーズ")

# confidenceとRMSDに相関があるか（confidenceが指標として機能しているかの確認）
from scipy.stats import spearmanr
rho, p = spearmanr([c for _, c, _ in rows], [r for _, _, r in rows])
print(f"\n   confidence と RMSD の順位相関 ρ = {rho:.2f}")
print("   （負の値ほど「confidenceが高いポーズほどRMSDが小さい」＝指標として機能）")

結晶JQ1:
   テンプレート照合: 中性 で成功

結晶構造に対するRMSD（重ね合わせなし・対称性考慮）
 rank  confidence   RMSD(Å)   判定
   10        0.26      1.51   成功(<2Å)
    1        0.63      0.62   成功(<2Å)
    2        0.57      0.51   成功(<2Å)
    3        0.52      0.59   成功(<2Å)
    4        0.50      0.67   成功(<2Å)
    5        0.50      0.54   成功(<2Å)
    6        0.46      0.58   成功(<2Å)
    7        0.45      1.49   成功(<2Å)
    8        0.39      1.46   成功(<2Å)
    9        0.36      1.48   成功(<2Å)

★ rank1 の RMSD = 0.62 Å
   10ポーズ中の最良 = rank2 の 0.51 Å
   2Å未満 = 10 / 10 ポーズ

   confidence と RMSD の順位相関 ρ = -0.75
   （負の値ほど「confidenceが高いポーズほどRMSDが小さい」＝指標として機能）


## 5. 3回目の実行：対照化合物

Val由来で、立体中心・重水素を含まず、サイズがtop_predと同程度の2化合物を実行する。

In [ ]:
# ===== 対照化合物2件（真の陰性 / モデルの失敗例）=====
# どちらも Val由来・立体中心なし・重水素なし・サイズは本命化合物と同程度
rows = [
    # 予測も実測も低い。モデルが「低活性」と正しく判断できている化合物
    ("neg_true", "brd4_bd1_simple.pdb",
     "Cc1cc(-c2nc3sc4c(c3c(=O)[nH]2)CCN(C)C4)cc(C)c1OCc1cccc(C(=O)Nc2ccccc2N)c1", ""),
    # 予測8.14に対し実測5.76。モデルがIC50換算で約240倍過大評価した化合物
    ("fp_highpred", "brd4_bd1_simple.pdb",
     "CCS(=O)(=O)Nc1cc(-c2cc(NCc3ccc(N(C)C)cc3)c(=O)n(C)c2)cc2c1ccn2C(C)(c1ccccn1)c1ccccn1", ""),
]
with open("controls_input.csv", "w") as f:
    f.write("complex_name,protein_path,ligand_description,protein_sequence\n")
    for r in rows:
        f.write(",".join(r) + "\n")

# 投入前にSMILESが読めるか確認（ここで落とす方が早い）
from rdkit import Chem
for name, _, smi, _ in rows:
    m = Chem.MolFromSmiles(smi)
    print(f"{name}: {'OK' if m else '★パース失敗'}  重原子={m.GetNumHeavyAtoms() if m else '-'}")
print()
print(open("controls_input.csv").read())

neg_true: OK  重原子=41
fp_highpred: OK  重原子=48

complex_name,protein_path,ligand_description,protein_sequence
neg_true,brd4_bd1_simple.pdb,Cc1cc(-c2nc3sc4c(c3c(=O)[nH]2)CCN(C)C4)cc(C)c1OCc1cccc(C(=O)Nc2ccccc2N)c1,
fp_highpred,brd4_bd1_simple.pdb,CCS(=O)(=O)Nc1cc(-c2cc(NCc3ccc(N(C)C)cc3)c(=O)n(C)c2)cc2c1ccn2C(C)(c1ccccn1)c1ccccn1,



In [ ]:
!python inference.py \
    --config default_inference_args.yaml \
    --protein_ligand_csv controls_input.csv \
    --out_dir results/controls_run \
    --samples_per_complex 10

Generating ESM language model embeddings
Processing 1 of 1 batches (2 sequences)
0it [00:00, ?it/s]/content/DiffDock/DiffDock/datasets/parse_chi.py:91: RuntimeWarning: invalid value encountered in cast
  Y = indices.astype(int)
2it [01:01, 30.69s/it]


## 6. 全化合物のconfidenceのまとめ

「判断の目安」は(S)-JQ1の10ポーズから読み取ったもので、他の化合物にそのまま当てはまる保証はない。

In [ ]:
import os, re, glob

def scores(out_dir):
    res = {}
    if not os.path.isdir(out_dir): return res
    for cdir in sorted(os.listdir(out_dir)):
        d = os.path.join(out_dir, cdir)
        if not os.path.isdir(d): continue
        pairs = []
        for f in os.listdir(d):
            m = re.search(r"rank(\d+)_confidence([-\d.]+)\.sdf", f)
            if m: pairs.append((int(m.group(1)), float(m.group(2))))
        if pairs: res[cdir] = sorted(pairs)
    return res

allr = {}
for d in ["results/jq1_s_run", "results/brd4_run", "results/controls_run"]:
    allr.update(scores(d))

# 実測・予測pIC50の対応表（参照用。値は val_predictions.csv より）
meta = {
    "JQ1_control_S": ("陽性対照(結晶構造あり)", None, None),
    "top_pred":      ("本命: 予測高・実測高",    9.29, 9.37),
    "neg_true":      ("陰性: 予測低・実測低",    5.46, 5.46),
    "fp_highpred":   ("失敗例: 予測高・実測低",  8.14, 5.76),
    "JQ1_control":   ("JQ1 R体(不活性側)",      None, None),
}

print(f"{'化合物':<16}{'役割':<22}{'予測':>6}{'実測':>7}{'rank1':>8}{'10本の幅':>12}")
print("-"*76)
for name in ["JQ1_control_S","top_pred","fp_highpred","neg_true","JQ1_control"]:
    if name not in allr: continue
    p = allr[name]
    role, pred, act = meta.get(name, ("",None,None))
    hi = max(s for _,s in p); lo = min(s for _,s in p)
    print(f"{name:<16}{role:<22}"
          f"{(f'{pred:.2f}' if pred else '-'):>6}{(f'{act:.2f}' if act else '-'):>7}"
          f"{p[0][1]:>+8.2f}{f'{hi:+.2f} 〜 {lo:+.2f}':>14}")

print("\n【判断の目安】JQ1 S体の校正結果より")
print("  confidence 0.46以上 → 結晶構造とのRMSD 0.6Å程度")
print("  confidence 0.45以下 → RMSD 1.5Å程度")
print("  10本の幅が狭い = サンプリングが収束している（確信度が高い）")

化合物             役割                        予測     実測   rank1       10本の幅
----------------------------------------------------------------------------
JQ1_control_S   陽性対照(結晶構造あり)               -      -   +0.63 +0.63 〜 +0.26
top_pred        本命: 予測高・実測高             9.29   9.37   -0.39 -0.39 〜 -3.61
fp_highpred     失敗例: 予測高・実測低            8.14   5.76   -1.10 -1.10 〜 -5.10
neg_true        陰性: 予測低・実測低             5.46   5.46   -2.60 -2.60 〜 -4.55
JQ1_control     JQ1 R体(不活性側)               -      -   -1.95 -1.95 〜 -3.19

【判断の目安】JQ1 S体の校正結果より
  confidence 0.46以上 → 結晶構造とのRMSD 0.6Å程度
  confidence 0.45以下 → RMSD 1.5Å程度
  10本の幅が狭い = サンプリングが収束している（確信度が高い）
